# Hardware Test in Pynq-Z2 Board

In [45]:
from pathlib import Path
from typing import Tuple

import numpy as np

from axi_stream_driver import NeuralNetworkOverlay

## Data

In [46]:
test_data = np.load("test_raw.npz", allow_pickle=True)

X_test = test_data["data"]
y_test_label = test_data["labels"]

X_test = X_test[:1000]
y_test_label = y_test_label[:1000]

label_map = {"lateral_L": 0, "lateral_R": 1, "supine": 2}
y_test = np.vectorize(label_map.get)(y_test_label)

print("\nDataset shapes:")
print(f"  Test:  {X_test.shape}, labels: {y_test.shape}")


Dataset shapes:
  Test:  (10000, 64), labels: (10000,)


### Quantization

In [47]:
# ap_ufixed<7,8>: scale factor = 2^(8-7) = 2
INPUT_SCALE = 2.0
# ap_fixed<14,9>: scale factor = 2^(9-16) = 2^-7 = 1/128
OUTPUT_SCALE = 1 / 128

encode_v = np.vectorize(lambda xi: np.uint16(np.round(xi / INPUT_SCALE)))
decode_v = np.vectorize(lambda yi: np.float32(yi) * OUTPUT_SCALE)

# encode: float32 -> uint16 raw bit pattern (lower 7 bits are ap_ufixed<7,8>)
# decode: int32 -> float32 (lower 14 bits are ap_fixed<14,9>)

## Neural Network

In [48]:
# Instantiate overlay for single or batch inference (64 in, 3 out)
nn = NeuralNetworkOverlay(
    "myproject.bit",
    x_shape=(64,),
    y_shape=(3,),
    input_dtype=np.uint16,   # 16-bit AXI-Stream for ap_ufixed<7,8>
    output_dtype=np.int32,   # 32-bit AXI-Stream for ap_fixed<14,9>
)

# Run inference on the full test set in one call
y_hw, dts, rate = nn.predict(
    X_test, profile=True, encode=encode_v, decode=decode_v
)

HW inference: 100%|██████████| 10000/10000 [00:11<00:00, 905.48it/s]


Classified 10000 samples in 91.242108 seconds (109.60 inferences / s)


### Evaluate

In [49]:
# Compute predicted classes from raw logits
y_pred_hw = np.argmax(y_hw, axis=1)

accuracy = np.mean(y_pred_hw == y_test)
print(f"Hardware accuracy: {accuracy*100:.2f}%")
print(f"\nFirst 10 logits:\n{y_hw[:10]}")
print(f"\nFirst 10 predictions: {y_pred_hw[:10]}")
print(f"First 10 ground truth:  {y_test[:10]}")

Hardware accuracy: 84.85%

First 10 logits:
[[ 0.765625   0.1171875 -0.7421875]
 [ 5.2734375 -5.046875   1.84375  ]
 [-2.1015625  1.0546875  3.046875 ]
 [-3.390625   3.03125    1.2265625]
 [-2.3671875  0.734375   1.28125  ]
 [-2.6953125 -0.515625   4.609375 ]
 [-0.84375    3.234375  -1.9921875]
 [ 0.28125   -5.1796875  5.65625  ]
 [-1.7421875  1.84375   -0.09375  ]
 [ 0.78125   -6.9921875  2.5234375]]

First 10 predictions: [0 0 2 1 2 2 1 2 1 2]
First 10 ground truth:  [0 0 2 1 1 2 1 0 1 0]


## Save

In [50]:
np.savez("y_hw.npz", logits=y_hw, predictions=y_pred_hw)